<!-- SPDX-License-Identifier: GPL-2.0-only -->
# 90 Quirks

The notebook reads the register `Documentation/quirks.md` back and proves it. The register lists
every quirk, bug and inconsistency of 3D-Baukasten 1.3 with what version 2.0 does with it, and its
column Test names the notebook cell that proves the row as `nn:name`. The notebook checks that the
register parses, that every tag resolves to a passed check of the executed notebooks under `out/`,
which is why `tests/run.py` runs it last, that the behaviour 2.0 keeps and fixes holds on the side
of the engine, that the rows of the interface which the notebooks 60 to 80 leave open hold in the
page of both looks, that the errata of the thesis are documented only, and the facts of the
repository, among them the rule that the files of 1.3 carry no licence.

A tag proves a row when its code cell ran without an error, a passed check names one of the IDs of
the row, and no failed check names one. The tags of this notebook resolve in its last cell, from the
lines its own tagged cells printed.

In [ ]:
width = 1440
height = 900
look = 'modern'

In [ ]:
import sys, json, math, re
sys.path.insert(0, '..')
from harness import *

In [ ]:
import contextlib
import io

#: The lines of the passed and the failed checks of each tagged cell of this notebook, by tag.
PROVED = {}


class Tee(io.TextIOBase):
    """Writes to the output of the cell and keeps a copy of every line."""

    def __init__(self, out, lines):
        self.out, self.lines, self.rest = out, lines, ''

    def write(self, text):
        self.out.write(text)
        self.rest += text
        *done, self.rest = self.rest.split('\n')
        self.lines.extend(done)
        return len(text)

    def flush(self):
        self.out.flush()


@contextlib.contextmanager
def proving(tag):
    """Keeps the lines a tagged cell prints under its tag, for the coverage of the tags 90:* in the last cell."""
    lines = PROVED.setdefault(tag, [])
    old = sys.stdout
    sys.stdout = Tee(old, lines)
    try:
        yield
    except Exception:
        lines.append('FAIL the cell raised')
        raise
    finally:
        sys.stdout = old


import nbformat

ID = r'(?:F4-Q\d+|Q-\d+|M-\d+|T-p\d+[a-z]?|C-\d+|R-\d+)'


def names(message, ids):
    """Answers whether the message of a check names one of the IDs `ids` as a whole word."""
    return any(re.search(rf'(?<![\w-]){re.escape(i)}(?![\w])', message) for i in ids)


def executed_cells(prefix):
    """Answers the tagged code cells of the executed notebook out/<prefix>_*.ipynb as {tag: (error, lines)}."""
    books = sorted(OUT.glob(f'{prefix}_*.ipynb'))
    if not books:
        return None
    cells = {}
    for cell in nbformat.read(books[0], as_version=4).cells:
        if cell.cell_type != 'code':
            continue
        text, error = '', False
        for output in cell.get('outputs', []):
            if output.get('output_type') == 'stream':
                text += output.get('text', '')
            elif output.get('output_type') == 'error':
                error = True
        for tag in cell.metadata.get('tags', []):
            if tag != 'parameters':
                cells[tag] = (error, text.splitlines())
    return cells


def proves(error, lines, ids):
    """Answers whether a cell proves a row: it ran without an error, a passed check names one of the IDs and no failed check names one."""
    if error or any(line.startswith('FAIL ') and names(line[5:], ids) for line in lines):
        return False
    return any(line.startswith('ok   ') and names(line[5:], ids) for line in lines)

## 1. The register parses

Every row of a table whose first column is ID has the five columns ID, Version 1.3, Version 2.0,
Status and Test. The IDs are unique over the whole register, the quirks of the notes run without a
gap, Q-1 to Q-112, F4-Q1 to F4-Q100 and M-1 to M-80, every status is one of the four words, and
every Test cell is the dash or tags `nn:name` separated by commas.

In [ ]:
ROWS = quirks_table()
check(len(ROWS) >= 300, 'the register has its rows', len(ROWS))
check(all(set(['ID', 'Version 1.3', 'Version 2.0', 'Status', 'Test']) <= set(r) for r in ROWS), 'every row has the five columns')
ids = [i for r in ROWS for i in r['ids']]
check(len(ids) == len(set(ids)), 'the IDs are unique over the register', [i for i in set(ids) if ids.count(i) > 1])
check(all(re.fullmatch(ID, i) for i in ids), 'every ID carries one of the prefixes Q, F4-Q, M, T-p, C and R', [i for i in ids if not re.fullmatch(ID, i)])
for prefix, last in (('Q-', 112), ('F4-Q', 100), ('M-', 80)):
    numbers = {int(i[len(prefix):]) for i in ids if i.startswith(prefix) and i[len(prefix):].isdigit()}
    check(numbers == set(range(1, last + 1)), f'the quirks {prefix}1 to {prefix}{last} without a gap', sorted(set(range(1, last + 1)) - numbers))
STATUS = {'Kept', 'Fixed', 'Changed on purpose', 'Not applicable'}
check(all(r['Status'] in STATUS for r in ROWS), 'every status is one of the four words', {r['Status'] for r in ROWS} - STATUS)
TAG = re.compile(r'\d\d:[a-z0-9-]+')
check(all(r['Test'] == '–' or all(TAG.fullmatch(t) for t in r['Test'].split(', ')) for r in ROWS), 'every Test cell is the dash or tags nn:name', [r['ID'] for r in ROWS if not (r['Test'] == '–' or all(TAG.fullmatch(t) for t in r['Test'].split(', ')))])
TAGGED = {}
for r in ROWS:
    if r['Test'] != '–':
        for t in r['Test'].split(', '):
            TAGGED.setdefault(t, []).append(r)
print(f'{len(TAGGED)} tags prove {sum(1 for r in ROWS if r["Test"] != "–")} rows')

## 2. Coverage

Every tag of the notebooks 00 to 80 resolves to its cell in the executed copy under `out/`, and that
cell proves every row that names the tag. The tags of this notebook follow in its last cell, after
the sections that prove them.

In [ ]:
OPEN_TAGS = {}
executed = {}
for tag, rows in sorted(TAGGED.items()):
    prefix, name = tag.split(':')
    if prefix == '90':
        continue
    if prefix not in executed:
        executed[prefix] = executed_cells(prefix)
    cells = executed[prefix]
    if cells is None:
        OPEN_TAGS[tag] = ['the notebook has not run']
    elif name not in cells:
        OPEN_TAGS[tag] = ['no cell carries the tag']
    else:
        error, lines = cells[name]
        missing = [r['ID'] for r in rows if not proves(error, lines, r['ids'])]
        if missing:
            OPEN_TAGS[tag] = missing
for tag, why in OPEN_TAGS.items():
    print(f'open {tag}: {why}')
done = sum(1 for t in TAGGED if not t.startswith('90:')) - len(OPEN_TAGS)
print(f'{done} of {sum(1 for t in TAGGED if not t.startswith("90:"))} tags of the notebooks 00 to 80 resolve, the last cell checks them')

## 3. Kept behaviour, engine side

The engine of the worker, booted on the build host, keeps what 1.3 did: the zoom ladder of factor
1.2 between its limits, the 43 textures in the order of `Fill_CB3` of 1.3, the room of `Def_Pov`
with its sky, light and three planes, the unit body with its origin at a corner and no export of the
raster, which is no item.

In [ ]:
service = Engine()
check(ZOOM_LADDER == [round(v, 3) for v in ladder(10.0)] and ladder(10.0)[0] < 5 and ladder(10.0)[-1] > 50, 'Q-28 the factor 1.2 and the two limits of 1.3 stay, so each end passes its limit by one step', ladder(10.0))
import subprocess
intact = subprocess.run(['git', 'show', '1c6982f:POV.bas'], cwd=ROOT, capture_output=True).stdout.decode('latin-1')
fill = re.findall(r'CB3\.AddItem "([^"]+)"', intact)
textures = [t['name'] for t in service.data('{ textures { name } }')['textures']]
check(textures == fill and len(textures) == 43, 'the 43 textures in the order of 1.3', [t for t in fill if t not in textures])
scene = service.data('mutation { createScene { id } }')['createScene']['id']
pov = lambda: service.data('query($s: ID!) { scene(id: $s) { pov(eye: { x: 15, y: 8, z: -15 }, target: { x: 0, y: 0, z: 0 }) } }', {'s': scene})['scene']['pov']
empty = pov()
check('// Objekt' not in empty, 'the raster is no item and is never exported')
numbers = lambda line: [float(v) for v in re.findall(r'-?\d+(?:\.\d+)?', line)]
golden = (ROOT / 'ausgabe.pov').read_text(encoding='latin-1').splitlines()
room = {key: next(l for l in golden if l.lstrip().startswith(key)) for key in ('light_source', 'sky_sphere')}
check(numbers(next(l for l in empty.splitlines() if l.startswith('light_source'))) == numbers(room['light_source']) and 'Bright_Blue_Sky' in empty, 'M-41 the light and the sky of Def_Pov stay those of 1.3', room)
planes = [numbers(l)[:4] for l in empty.splitlines() if l.startswith('plane')]
check(planes == [numbers(l)[:4] for l in golden if l.startswith('plane')], 'M-41 the floor and the two walls stay those of 1.3', planes)
service.data('mutation($s: ID!) { addItem(scene: $s, input: { kind: CUBE }) { id } }', {'s': scene})
check('box { <0, 0, -1>, <1, 1, 0>' in pov(), 'the unit cube has its origin at a corner, as the box of 1.3', pov()[-200:])

## 4. Fixed behaviour, engine side

The export writes a declared body under an identifier POV-Ray accepts, with the prefix `Neu_` and
the umlauts spelled out, on its own line ahead of the items, places it with a scale of its own group
and not zero, wraps the pigments of `textures.inc` into a pigment, and closes each group of the CSG
with its own block. A body without a colour gets the grey of the service.

In [ ]:
added = service.data('mutation($s: ID!) { addItem(scene: $s, input: { kind: SPHERE }) { color } }', {'s': scene})['addItem']
check(added['color'] == '#d9d9d9', 'R-51 addItem gives a body without a colour the grey #d9d9d9', added)
declared = service.data('mutation($s: ID!) { addItem(scene: $s, input: { kind: DECLARE, name: "Werkstück 1" }) { id } }', {'s': scene})['addItem']['id']
service.data('mutation($s: ID!, $p: ID) { addItem(scene: $s, input: { kind: CONE, parent: $p }) { id } }', {'s': scene, 'p': declared})
service.data('mutation($s: ID!, $d: ID) { addItem(scene: $s, input: { kind: INSTANCE, source: $d }) { id } }', {'s': scene, 'd': declared})
text = pov()
check(re.search(r'^#declare Neu_Werkstueck_?1 = union \{$', text, re.M) is not None, 'M-36 the identifier is Neu_ and the name with the umlaut spelled out, M-35 on its own line', [l for l in text.splitlines() if 'declare' in l])
check('scale <0, 0, 0>' not in text and re.search(r'object \{ Neu_Werkstueck_?1\n\s*scale <1, 1, 1>', text) is not None, 'M-38 the placement takes the scale of its own group, never zero', text[text.find('object {'):][:80])
service.data('mutation($s: ID!) { addItem(scene: $s, input: { kind: SPHERE, texture: "Jade" }) { id } }', {'s': scene})
service.data('mutation($s: ID!) { addItem(scene: $s, input: { kind: SPHERE, texture: "Gold_Metal" }) { id } }', {'s': scene})
text = pov()
check('texture { pigment { Jade } }' in text and 'texture { Gold_Metal }' in text, 'M-43 the export wraps the pigment Jade and leaves the texture Gold_Metal', [l.strip() for l in text.splitlines() if 'texture' in l])
group = service.data('mutation($s: ID!) { addItem(scene: $s, input: { kind: UNION }) { id } }', {'s': scene})['addItem']['id']
inner = service.data('mutation($s: ID!, $p: ID) { addItem(scene: $s, input: { kind: DIFFERENCE, parent: $p }) { id } }', {'s': scene, 'p': group})['addItem']['id']
for kind in ('CUBE', 'CYLINDER'):
    service.data('mutation($s: ID!, $p: ID, $k: Kind!) { addItem(scene: $s, input: { kind: $k, parent: $p }) { id } }', {'s': scene, 'p': inner, 'k': kind})
text = pov()
check(text.count('{') == text.count('}') and re.search(r'union \{\s*(//[^\n]*\n\s*)?difference \{', text) is not None, 'the nested groups close each its own block', text[text.find('union {', text.find('Objekt')):][:200])
service.stop()

## 5. Rows of the interface that 60 to 80 leave open

One cell per scenario of the register, in the page at 1440 by 900, in the modern look and where the
row differs in the classic look as well: Ändern updates the list and leaves the raster alone, typing
a name into CB1 sets its mask, an empty or unknown name inserts nothing, Würfel follows the last
delete with its mask, `#Neu` inserts no CSG-Summe, Übersetzen closes an open group and says so, ×
hides F4 with its state intact, the inner radius of a disc follows the selection, a field that holds
no number is refused, `Neu_Ende` alone does nothing, a choice in CB1 starts the marking, the modes
Position and Bearbeiten drag bodies and handles, a quick drag included that lets go before the
worker answers the pick, new bodies take the colours of the palette in turn, the modern bar names
the view under the pointer, the modern look shows a crosshair while a body is placed, and a press on
a body in Position mode selects it.

In [ ]:
site = Site()
page = Browser(site.url, width=width, height=height, look=look)
page.open()
DEFERRED = []


def expect(condition, message, *context):
    """Checks as check() does, and keeps a failure for the last cell instead of stopping here."""
    try:
        check(condition, message, *context)
    except Failure as failure:
        DEFERRED.append(str(failure))
        print(f"FAIL {message}")


def fresh(look=None):
    """Empties the scene through Datei › Neu and answers Ja, as a user does."""
    page.eval("() => { bausteine.act('neu'); }")
    page.settle()
    if page.eval("() => document.querySelector('#msgbox').open"):
        page.page.click('#msgbox .c-default')
        page.settle()
    page.key('Escape')
    page.key('Escape')


check(page.errors() == [], 'the page starts without an error', page.errors())


def at(view, x, y, z):
    """Answers the pixel of the page that shows the point (x, y, z) in the view `view`."""
    return page.eval(
        "([v, x, y, z]) => { const r = document.querySelector(`.view[data-view='${v}']`).getBoundingClientRect();"
        " const b = bausteine.app.bridge; const p = b.project(v, x, y, z); if (!p) return null;"
        " const w = b.cameras ? b.cameras[v * 18 + 16] : 0; const inset = w > 0 ? Math.max(0, (r.width - w) / 2) : 0;"
        " return [r.left + inset + p[0], r.top + inset + p[1]]; }", [view, x, y, z])


def hover_world(view, x, y, z):
    """Moves the pointer over the point (x, y, z) of the view `view`."""
    p = at(view, x, y, z)
    page.page.mouse.move(p[0], p[1])
    page.settle()
    return p

In [ ]:
with proving('aendern-list'):
    fresh()
    page.page.click('#tb2 [data-body="Prisma"]'); page.page.click('#btn-einf'); page.settle()
    page.page.click('#cb2 [data-line="1"]'); page.settle()
    page.entry('Kugel'); page.page.fill('#t1-1', '3')
    page.page.click('#btn-change'); page.settle()
    check(page.lines() == ['0 Raster', '1 Kugel'] and page.eval("() => bausteine.app.line") == 1, 'F4-Q37 Ändern updates the text of the list and keeps the selection', page.lines())

In [ ]:
with proving('aendern-raster'):
    page.key('Escape'); page.key('Escape')
    check(page.eval("() => bausteine.app.line") == -1 and page.eval("() => document.querySelector('#btn-change').disabled && document.querySelector('#btn-clear').disabled"), 'F4-Q34 with nothing selected Ändern and Löschen act on nothing')
    page.page.click('#cb2 [data-line="0"]'); page.settle()
    check(page.eval("() => document.querySelector('#btn-change').disabled && document.querySelector('#btn-clear').disabled"), 'F4-Q35 the raster cannot be changed or deleted')
    n = page.eval("() => bausteine.app.scene.items.length")
    page.page.locator('#views').hover(); page.eval("() => document.activeElement && document.activeElement.blur()")
    page.key('Delete'); page.settle()
    check(page.eval("() => bausteine.app.scene.items.length") == n, 'F4-Q35 Entf on the raster removes nothing, since the raster is no item')

In [ ]:
with proving('cb1-typing'):
    page.key('Escape'); page.key('Escape')
    page.eval("() => { const e = document.querySelector('#cb1-edit'); e.value = 'kegel'; e.dispatchEvent(new Event('input')); }"); page.settle()
    check(page.texts('#l1-1') == ['Mittelpunkt der Grundfläche (x,y,z)'] and page.texts('#l1-6') == ['Höhe'] and page.eval("() => document.querySelector('#cb1-edit').value") == 'Kegel', 'F4-Q42 typing a known name into CB1 sets its mask', page.texts('#l1-1'))

In [ ]:
with proving('empty-name'):
    page.page.click('#cb2 [data-line="1"]'); page.settle()
    before = page.lines()
    page.graphql_log()
    page.eval("() => { document.querySelector('#cb1-edit').value = ''; }")
    page.page.click('#btn-einf'); page.settle()
    check(page.lines() == before, 'F4-Q58 an empty name inserts nothing', page.lines())
    page.entry('Klotz'); page.page.click('#btn-einf'); page.settle()
    check(page.lines() == before and page.status().startswith('Unbekannter Körper'), 'F4-Q58 an unknown name inserts nothing and says so', page.status())
    check(not any('addItem' in e['query'] or 'moveItem' in e['query'] for e in page.graphql_log()), 'F4-Q40 the page checks the name before anything changes')

In [ ]:
with proving('default-body'):
    page.entry('Quader')
    check(page.eval("() => document.querySelector('#cb1-edit').value") == 'Quader', 'F4-Q27 the body after the start is Quader')
    fresh()
    page.page.click('#tb2 [data-body="Kugel"]'); page.page.click('#btn-einf'); page.settle()
    page.page.click('#cb2 [data-line="1"]'); page.settle()
    page.page.click('#btn-clear'); page.settle()
    check(page.eval("() => document.querySelector('#cb1-edit').value") == 'Würfel', 'F4-Q27 after the last delete CB1 becomes Würfel', page.eval("() => document.querySelector('#cb1-edit').value"))
    check(page.texts('#l1-1') == ['Eckpunkt (hinten,unten,links) (x,y,z)'] and page.texts('#l1-4') == ['Kantenlänge'] and page.fields()[4] is None, 'F4-Q28 and the mask follows', page.texts('#l1-1'), page.texts('#l1-4'))

In [ ]:
with proving('neu-merge'):
    fresh()
    page.entry('#Neu'); page.page.click('#btn-einf'); page.settle()
    page.page.click('#tb2 [data-body="Würfel"]'); page.page.click('#btn-einf'); page.settle()
    page.entry('Neu_Ende'); page.page.click('#btn-einf'); page.settle()
    kinds = page.eval("() => bausteine.app.scene.items.map((i) => i.kind)")
    check(kinds == ['DECLARE', 'CUBE'] and 'MERGE' not in kinds, 'F4-Q50 #Neu inserts a declaration group and no CSG-Summe', kinds)

In [ ]:
def end_check(look_name):
    """Opens a group, lets Übersetzen close it, and answers what the page showed and posted."""
    page.look(look_name); page.settle()
    fresh()
    page.page.click('#sstab1 [data-tab="koerper"]'); page.settle()
    page.entry('CSG-Differenz'); page.page.click('#btn-einf'); page.settle()
    page.page.click('#tb2 [data-body="Würfel"]'); page.page.click('#btn-einf'); page.settle()
    page.page.click('#sstab1 [data-tab="ausgabe"]'); page.settle()
    page.graphql_log()
    page.page.click('#bpov-comp'); page.settle()
    shown = page.eval("() => [document.querySelector('#msgbox').open, document.querySelector('#msgbox-text').textContent, document.querySelector('#msgbox-title').textContent, !document.querySelector('#toast').hidden && document.querySelector('#toast').textContent]")
    if shown[0]:
        page.page.click('#msgbox .c-default'); page.settle()
    posted = [e['query'] for e in page.graphql_log()]
    selected = page.eval("() => bausteine.app.lines[bausteine.app.line]")
    page.page.click('#sstab1 [data-tab="koerper"]'); page.settle()
    return shown, posted, selected


with proving('end-check'):
    shown, posted, selected = end_check('modern')
    check(not shown[0] and 'CSG-Ende' in (shown[3] or ''), 'F4-Q54 Übersetzen with an open group says so in the modern look', shown)
    check(selected['end'] is True and not any('addItem' in q or 'updateItem' in q for q in posted), 'F4-Q54 the group closes and nothing is added to the scene', selected, posted)
    shown, posted, selected = end_check('classic')
    check(shown[:3] == [True, 'CSG_Ende fehlt - wird angefügt ', ''], 'F4-Q54 the classic look shows the message box of 1.3 with its empty title', shown)
    check(selected['end'] is True and not any('addItem' in q for q in posted), 'F4-Q54 and the group closes there too', selected)
    page.look('modern'); page.settle()

In [ ]:
with proving('close-f4'):
    page.look('classic'); page.settle()
    lines = page.lines()
    page.page.click('#f4 .win-close'); page.settle()
    check(not page.visible('#f4'), 'Q-85 the × of the classic F4 hides it')
    page.menu('ansicht/fenster-zeigen'); page.settle()
    check(page.visible('#f4') and page.lines() == lines and len(page.texts('#cb3 option')) == 44, 'Q-85 Alle Fenster zeigen shows it again with its objects and the 43 textures once', page.lines())
    page.look('modern'); page.settle()
    check(not any(page.visible(f'{w} .win-close') for w in ('#f2', '#f3', '#f4', '#f6')), 'Q-85 the modern panels have no ×')

In [ ]:
with proving('disc-inner'):
    fresh()
    page.entry('Disc'); page.page.fill('#t1-4', '3'); page.page.fill('#t1-10', '1')
    page.page.click('#btn-einf'); page.settle()
    page.entry('Kugel'); page.settle()
    page.entry('Disc')
    check(page.eval("() => document.querySelector('#t1-10').value") == '0', 'F4-Q61 the inner radius resets with the mask', page.eval("() => document.querySelector('#t1-10').value"))
    page.page.click('#cb2 [data-line="1"]'); page.settle()
    check(page.eval("() => document.querySelector('#t1-10').value") == '1', 'F4-Q61 and shows the inner radius of the selected disc', page.eval("() => document.querySelector('#t1-10').value"))

In [ ]:
with proving('validation'):
    page.key('Escape'); page.key('Escape')
    page.entry('Kegel')
    page.graphql_log()
    n = page.eval("() => bausteine.app.scene.items.length")
    page.page.fill('#t1-6', 'abc'); page.page.click('#btn-einf'); page.settle()
    posted = [e['query'] for e in page.graphql_log()]
    expect(page.eval("() => bausteine.app.scene.items.length") == n and not any('addItem' in q for q in posted) and page.status() != '', 'F4-Q63 Einfügen refuses a field that holds no number with a status line and sends nothing', page.status(), posted)
    if page.eval("() => bausteine.app.scene.items.length") != n:
        page.key('Control+z'); page.settle()
    service_refusal = page.eval("async () => (await bausteine.graphql('mutation($s: ID!) { addItem(scene: $s, input: { kind: CUBE, size: { x: 20000, y: 1, z: 1 } }) { id } }', { s: bausteine.app.scene.id }))")
    check('errors' in json.dumps(service_refusal), 'F4-Q63 the service takes only sizes up to 10000', service_refusal)

In [ ]:
with proving('neu-ende-alone'):
    fresh()
    page.eval("() => { document.querySelector('#tstatus').value = ''; }")
    page.entry('Neu_Ende'); page.page.click('#btn-einf'); page.settle()
    check(page.lines() == ['0 Raster'], 'F4-Q49 Neu_Ende without a declaration does nothing', page.lines())

In [ ]:
with proving('cb1-reset'):
    page.page.click('#tb2 [data-body="Quader"]'); page.settle()
    page.click_world(TOP, 2, 3, 0)
    check(page.fields()[:2] == ['2*', '3*'], 'the marking locks x and y')
    page.page.click('#cb1 .c-combo-arrow'); page.settle()
    page.page.click('#cb1 .c-combo-list [data-entry="Kugel"]'); page.settle()
    state = page.eval("() => [bausteine.app.mode, bausteine.mark().phase, bausteine.mark().locked, document.querySelectorAll('.t1.locked').length]")
    check(state == ['einfuegen', 1, [], 0] and page.status() == 'Bezugspunkt in einem Riss markieren', 'F4-Q66 a choice in CB1 starts the marking as the body bar does and frees the fields', state, page.status())
    page.key('Escape'); page.key('Escape')

In [ ]:
with proving('bearbeiten-mode'):
    fresh()
    page.page.click('#tb2 [data-body="Quader"]'); page.settle()
    for f, v in (('#t1-1', '2'), ('#t1-2', '3'), ('#t1-4', '2'), ('#t1-5', '2'), ('#t1-6', '2')):
        page.page.fill(f, v)
    page.page.click('#btn-einf'); page.settle()
    page.key('Escape')
    page.page.click('#tb1 [data-tool="1"]'); page.settle()
    start, end = at(TOP, 3, 4, 0), at(TOP, 5, 6, 0)
    # The press asks the worker which body lies under it, and the drag moves the body once it knows.
    page.page.mouse.move(*start); page.page.mouse.down(); page.settle()
    page.page.mouse.move(*end, steps=8); page.page.mouse.up(); page.settle()
    moved = page.eval("() => bausteine.app.scene.items[0]")
    check(moved['position']['x'] == 4 and moved['position']['y'] == 5 and moved['size']['x'] == 2, 'T-p79b the tool Position drags a body in the plane of the view', moved['position'])
    # The worker answers every pick 300 ms late, and a quick drag lets go long before the answer.
    LATE = "() => { const b = bausteine.app.bridge; const pick = Object.getPrototypeOf(b).pick; b.late = 0; b.pick = (...a) => { b.late += 1; return new Promise((r) => setTimeout(r, 300)).then(() => pick.apply(b, a)).finally(() => { b.late -= 1; }); }; }"
    PICKED = "() => bausteine.app.bridge.late === 0"

    def quick(start, end):
        """Presses at `start`, moves to `end` in two steps and lets go at once, then waits for the late pick."""
        page.page.mouse.move(*start); page.page.mouse.down()
        page.page.mouse.move(*end, steps=2); page.page.mouse.up()
        page.until(PICKED)

    page.eval(LATE)
    quick(at(TOP, 5, 6, 0), at(TOP, 7, 8, 0))
    late = page.eval("() => bausteine.app.scene.items[0].position")
    check(late['x'] == 6 and late['y'] == 7, 'T-p79b a quick drag that lets go before the pick answers moves the body the whole distance', late)
    check(page.eval("() => document.querySelector('.view.dragging') === null"), 'T-p79b no view keeps the class dragging after the quick drag')
    page.eval("() => { bausteine.act('rueckgaengig'); }"); page.settle()
    check(page.eval("() => bausteine.app.scene.items[0].position") == moved['position'], 'T-p79b one Rückgängig takes the whole quick drag back', page.eval("() => bausteine.app.scene.items[0].position"))
    centre = page.eval("() => ({ ...bausteine.app.center })")
    quick(at(TOP, -6, -6, 0), at(TOP, -4, -6, 0))
    panned = page.eval("() => ({ ...bausteine.app.center })")
    check(abs(panned['x'] - centre['x'] + 2) < 0.05 and abs(panned['y'] - centre['y']) < 0.05, 'T-p79b the same quick drag over empty space pans the view the whole distance', centre, panned)
    page.key('Escape')
    page.page.mouse.click(*at(TOP, 5, 6, 0)); page.until(PICKED)
    chosen = page.eval("() => [bausteine.app.lines[bausteine.app.line]?.item ?? null, bausteine.app.scene.items[0].id, bausteine.app.scene.items[0].position]")
    check(chosen[0] == chosen[1] and chosen[2] == moved['position'], 'T-p79b a click with the late pick selects the body and moves nothing', chosen)
    page.eval("() => { const b = bausteine.app.bridge; delete b.pick; delete b.late; }")
    page.page.click('#cb2 [data-line="1"]'); page.settle()
    page.page.click('#tb1 [data-tool="3"]'); page.settle()
    check(page.eval("() => bausteine.app.mode") == 'bearbeiten', 'Q-78 Bearbeiten is a mode')
    corner, further = at(TOP, 6, 7, 0), at(TOP, 7, 8, 0)
    page.page.mouse.move(*corner); page.settle()
    page.page.mouse.down(); page.page.mouse.move(*further, steps=8); page.page.mouse.up(); page.settle()
    sized = page.eval("() => bausteine.app.scene.items[0]")
    check(sized['size']['x'] == 3 and sized['size']['y'] == 3 and sized['position'] == moved['position'], 'Q-78 the handles of Bearbeiten change the size of the selected body', sized['size'])
    page.page.click('#tb1 [data-tool="2"]'); page.settle(); page.key('Escape')

In [ ]:
with proving('palette'):
    fresh()
    for _ in range(11):
        page.page.click('#tb2 [data-body="Würfel"]'); page.page.click('#btn-einf'); page.settle()
    colours = page.eval("() => bausteine.app.scene.items.map((i) => i.color.toLowerCase())")
    check(len(set(colours[:10])) == 10 and colours[10] == colours[0], 'F4-Q31 new bodies take ten colours in turn', colours)
    check('#000000' not in colours and '#d9d9d9' not in colours, 'R-51 the page passes the next colour of its palette, never black and not the grey of the service', colours)

In [ ]:
with proving('view-name'):
    page.key('Escape'); page.key('Escape')
    hover_world(TOP, 3, 4, 0)
    modern_name = page.texts('#label1-f5')
    check(modern_name == ['Grund'] and page.visible('#label1-f5'), 'Q-24 the modern bar names the view under the pointer, Q-90 in Label1 of F5', modern_name)
    page.look('classic'); page.settle()
    hover_world(TOP, 3, 4, 0)
    check(page.texts('#label1-f5') in ([''], []) or not page.visible('#label1-f5'), 'Q-24 the classic bar shows no name, Q-90 and hides Label1', page.texts('#label1-f5'))
    page.look('modern'); page.settle()

In [ ]:
with proving('cursor'):
    page.page.click('#tb2 [data-body="Kugel"]'); page.settle()
    hover_world(TOP, 1, 1, 0)
    modern_cursor = page.css('.view[data-view="2"]', 'cursor')
    page.key('Escape')
    page.look('classic'); page.settle()
    page.page.click('#tb2 [data-body="Kugel"]'); page.settle()
    hover_world(TOP, 1, 1, 0)
    classic_cursor = page.css('.view[data-view="2"]', 'cursor')
    page.key('Escape')
    page.look('modern'); page.settle()
    check(modern_cursor == 'crosshair' and classic_cursor in ('default', 'auto'), 'Q-25 the modern look shows a crosshair while a body is placed, the classic look keeps the arrow', modern_cursor, classic_cursor)

In [ ]:
with proving('position-click'):
    fresh()
    page.page.click('#tb2 [data-body="Würfel"]')
    for f, v in (('#t1-1', '2'), ('#t1-2', '3'), ('#t1-4', '2')):
        page.page.fill(f, v)
    page.page.click('#btn-einf'); page.settle()
    page.key('Escape'); page.key('Escape')
    page.page.click('#tb1 [data-tool="1"]'); page.settle()
    page.page.mouse.click(*at(TOP, 3, 4, 0)); page.settle()
    check(page.eval("() => bausteine.app.line") == 1 and page.eval("() => bausteine.app.mode") == 'position', 'R-10 in Position mode a press on a body selects it', page.eval("() => bausteine.app.line"))
    page.page.keyboard.down('Alt'); page.page.mouse.click(*at(TOP, 7, 8, 0)); page.page.keyboard.up('Alt'); page.settle()
    check(page.fields()[:2] == ['7*', '8*'], 'R-10 Alt marks a point in Position mode', page.fields()[:2])
    page.key('Escape'); page.key('Escape')
    page.page.click('#tb1 [data-tool="2"]'); page.settle(); page.key('Escape')
    check(page.errors() == [], 'no error in the scenarios', page.errors())

## 6. The errata of the thesis are documented only

Every row of section 16 of the register, the thesis against the program, is `Not applicable` with
the dash, since 2.0 has no thesis to correct. The page Hilfe shows the samples of POV-Ray in their
corrected form, and the forms the thesis printed stand only in quotations marked as such.

In [ ]:
errata = [r for r in ROWS if r['section'].startswith('16.')]
check(errata and all(r['Status'] == 'Not applicable' and r['Test'] == '–' for r in errata), 'the rows of section 16 are Not applicable with the dash', [r['ID'] for r in errata if r['Status'] != 'Not applicable' or r['Test'] != '–'])
page.eval("() => { bausteine.act('hilfe'); }"); page.settle()
help_text = page.eval("() => document.querySelector('#help-dialog').textContent")
help_html = page.eval("() => document.querySelector('#help-dialog').innerHTML")
words = ['light_source', 'smooth_triangle', 'height_field', 'color_map', 'image_map', 'map_type', 'bump_map', 'bump_size', 'phong_size', 'look_at', 'plane { <0,1,0>, -5 }']
check(all(w in help_text for w in words), 'the page Hilfe shows the corrected samples of POV-Ray', [w for w in words if w not in help_text])
outside = re.sub(r'<q class="thesis">.*?</q>', '', help_html, flags=re.S).lower()
hyphenated = [w for w in ('light-source', 'smooth-triangle', 'height-field', 'color-map', 'image-map', 'map-type', 'bump-map', 'bump-size', 'phong-size', 'look-at') if w in outside]
check(hyphenated == [], 'the printed forms of the thesis stand only in quotations', hyphenated)
page.eval("() => document.querySelector('#help-dialog').close()")

## 7. Facts of the repository

The files of 1.3 stay at the root unchanged since the upstream head `9b91030`, without a licence.
`POV.bas` holds the replacement characters of its conversion, and the intact file of the commit
`1c6982f` the section sign as the name of a function. `3d_bau.vbp` points to an `F6.frm` that the
repository lacks, `3d_Form2.frm` names a `.frx` whose case differs from the file, `F5.frx` is a byte
for byte copy of `F3.frx`, and five example files are one scene. The licence lint checks the SPDX
line of every file of 2.0 and that no file, commit message or change of the history spells the name
of the permissive licence an earlier draft used. The pictures under `Documentation/images/` are the
renders the notebooks write under `out/`.

In [ ]:
with proving('repository'):
    git = lambda *args: subprocess.run(['git', *args], cwd=ROOT, capture_output=True)
    originals = [n for n in git('ls-tree', '--name-only', '9b91030').stdout.decode('utf-8').splitlines() if n != 'README.md']
    changed = git('diff', '--name-only', '9b91030', '--', *originals).stdout.decode('utf-8').split()
    check(len(originals) > 40 and changed == [], 'T-p0e the files of 1.3 stay unchanged since the upstream head, README.md excepted, which 2.0 writes', changed)
    check(not any((ROOT / n).exists() for n in ('LICENSE', 'LICENCE', 'LICENSE.txt')) and 'GPL' not in (ROOT / '3d_bau.vbp').read_text(encoding='latin-1'), 'T-p0e and carry no licence, none is assumed')
    pov_bas = (ROOT / 'POV.bas').read_text(encoding='utf-8')
    check(pov_bas.count('\ufffd') == 90 and sum('\ufffd' in line for line in pov_bas.splitlines()) == 72, 'M-22 POV.bas holds 90 replacement characters on 72 lines', pov_bas.count('\ufffd'))
    check(git('show', '1c6982f:POV.bas').stdout.count(b'\xa7') > 0, 'M-22 the intact POV.bas of 1c6982f holds the section sign')
    vbp = (ROOT / '3d_bau.vbp').read_text(encoding='latin-1')
    check('Form=..\\Programme\\DevStudio\\VB\\F6.frm' in vbp and not any(p.name.lower() == 'f6.frm' for p in ROOT.iterdir()), 'Q-80 3d_bau.vbp points to an F6.frm the repository lacks', [l for l in vbp.splitlines() if 'F6' in l])
    form2 = (ROOT / '3d_Form2.frm').read_text(encoding='latin-1')
    check('"3d_Form2.frx"' in form2 and (ROOT / '3D_FORM2.FRX').exists() and not any(p.name == '3d_Form2.frx' for p in ROOT.iterdir()), 'Q-72 3d_Form2.frm names 3d_Form2.frx, the file is 3D_FORM2.FRX')
    check((ROOT / 'F5.frx').read_bytes() == (ROOT / 'F3.frx').read_bytes(), 'Q-94 F5.frx is a byte for byte copy of F3.frx')
    copies = {(ROOT / n).read_bytes() for n in ('quader1.pov', 'quader2.pov', 'qyramid1.pov', 'qyramid1.pov.bak', 'qyramide1.pov')}
    check(len(copies) == 1, 'M-74 five example files are copies of one scene')
    lint = subprocess.run([sys.executable, '-I', str(ROOT / 'tools' / 'license_lint.py')], cwd=ROOT, capture_output=True, text=True)
    check(lint.returncode == 0 and lint.stdout.strip().endswith(' pass'), 'R-33 the licence lint passes over the files, the messages and the history', lint.stdout, lint.stderr)
    renders = {'zugeordnete-normalrisse.png': 'readme-modern.png', 'raytracing.png': 'readme-raytracing.png', 'klassisch.png': 'readme-klassisch.png'}
    images = ROOT / 'Documentation' / 'images'
    check(sorted(p.name for p in images.glob('*.png')) == sorted(renders), 'the pictures under Documentation/images are the three renders of README.md', sorted(p.name for p in images.glob('*.png')))
    from PIL import Image, ImageChops

    def alike(a, b):
        """Answers whether the pictures a and b are equal, or of one size with at most one pixel in a hundred visibly apart."""
        if not b.exists():
            return False
        if a.read_bytes() == b.read_bytes():
            return True
        one, two = Image.open(a).convert('RGB'), Image.open(b).convert('RGB')
        if one.size != two.size:
            return False
        apart = sum(1 for p in ImageChops.difference(one, two).convert('L').getdata() if p > 32)
        return apart <= one.size[0] * one.size[1] / 100

    same = {name: alike(images / name, OUT / out) for name, out in renders.items()}
    check(all(same.values()), 'the pictures are the renders that notebook 60 writes under out/', same)

## 8. Every tag resolves

The tags of the notebooks 00 to 80, gathered in section 2, and the tags of this notebook, proved by
its cells above, each prove every row that names them.

In [ ]:
for tag, rows in sorted(TAGGED.items()):
    if not tag.startswith('90:'):
        continue
    lines = PROVED.get(tag.split(':')[1])
    if lines is None:
        OPEN_TAGS[tag] = ['no cell carries the tag']
        continue
    missing = [r['ID'] for r in rows if not proves(False, lines, r['ids'])]
    if missing:
        OPEN_TAGS[tag] = missing
check(OPEN_TAGS == {}, f'every one of the {len(TAGGED)} tags of the register resolves to a passed check', *[f'{t}: {w}' for t, w in sorted(OPEN_TAGS.items())])

In [ ]:
page.close()
site.stop()
check(DEFERRED == [], 'every check that waited for the end holds', *DEFERRED)